In [110]:
import os
from getpass import getpass
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import FastEmbedSparse, RetrievalMode
from tqdm.notebook import tqdm
from tika import parser
from langchain_core.documents import Document
import json
import re

In [72]:
DATA_DIR = Path("data")
STATS_DIR = Path("stats")

CHAPITRE_PATTERN = r"(?=\bChapitre\s+\d+\b)"
SECTION_PATTERN = r"(?=\bSection\s+\d+\b)"
ARTICLE_PATTERN = r"(?=\b(?:Article\s+\d+|Art\.\s*\d+)\b)"

In [3]:
most_recent_file = max(
    (f for f in STATS_DIR.iterdir() if f.suffix == ".json"),
    key=lambda f: f.stat().st_mtime
)

with open(most_recent_file, "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
# TODO : indexer .docx et .txt plus tard

In [73]:
# creation de dicts pour optimiser la recherche du titre pour chaque document
# TODO : comment gerer le mistmatch entre les langues (voir BZc4iYDbPh4FufpSQ dans json) ?
doc_id_to_metadata = {}
lex_id_to_title = {}
for content, metadata_dict in data.items():
    ref = metadata_dict.get("refs")[0]
    lex_id = ref.get("lex_id")
    if metadata_dict.get("cat") == "summary":
        lex_id_to_title[lex_id] = content
    else:
        doc_id = metadata_dict.get("doc_id")
        doc_id_to_metadata[doc_id] = {"lex_id": lex_id, "source": metadata_dict.get("source")}

In [74]:
def clean_text(text, source):
    # TODO : a ameliorer
    cleaner_text = re.sub(r'\s+', ' ', text).strip()
    cleaner_text = re.sub(r"\.{4,}", " ", cleaner_text)
    if source == "fedlex":
        cleaner_text = re.sub(r'\b(?:RO \d{4} \d+|RS \d+(?:\.\d+)+)\b', '', cleaner_text)
    return cleaner_text

In [104]:
from langchain_text_splitters import CharacterTextSplitter

splitter_struct_aware = RecursiveCharacterTextSplitter(
    chunk_size=1000, # TODO : comment definir (dans ce cas nb caracteres et pas tokens) ?
    chunk_overlap=200, # TODO : comment definir (dans ce cas nb caracteres et pas tokens) ?
    separators=[
        SECTION_PATTERN,
        ARTICLE_PATTERN,
        "\n\n",
        "\n",
        " ",
        "",
    ],
    #length_function=count_nb_tokens, # TODO : si necessaire de compter en tokens alors modifier ici et modifier size + overlap
    is_separator_regex=True,
    keep_separator=True,
    add_start_index=True,
)

splitter_sliding_window = CharacterTextSplitter(
    chunk_size=6000,
    chunk_overlap=400,
    separator = ' ',
    add_start_index=True,
)

In [105]:
chunks = []

for file in tqdm(DATA_DIR.glob("*.pdf")):
    doc_id = file.stem
    lex_id = doc_id_to_metadata[doc_id]["lex_id"]
    source = doc_id_to_metadata[doc_id]["source"]
    summary = lex_id_to_title.get(lex_id) if lex_id else ""
    title = summary.split("\n")[0]
    parsed_pdf = parser.from_file(str(file))
    extracted_text = parsed_pdf['content']
    cleaner_text = clean_text(extracted_text, source)
    extracted_metadata = parsed_pdf['metadata']

    doc = Document(
        page_content=cleaner_text,
        metadata={
            "doc_id": doc_id,
            "filepath": str(file),
            "source": source,
            "total_pages": int(extracted_metadata["xmpTPg:NPages"]),
            "creationDate": extracted_metadata.get('xmp:CreateDate')
        },
    )

    chunks_from_struct = splitter_struct_aware.split_documents([doc])
    chunks_from_window = splitter_sliding_window.split_documents([doc])

    for chunk in chunks_from_struct:
        page_content = f"Document: {title}\n{chunk.page_content}" if title else chunk.page_content
        metadata = {**chunk.metadata, "chunking": "from document structure"}
        chunks.append(
            Document(
                page_content=page_content,
                metadata=metadata # TODO : ajouter reference aux articles contenus dans ce chunk + lex id, langue et type ?
            )
        )

    for chunk in chunks_from_window:
        page_content = f"Document: {title}\n{chunk.page_content}" if title else chunk.page_content
        metadata = {**chunk.metadata, "chunking": "with sliding window"}
        chunks.append(
            Document(
                page_content=page_content,
                metadata=metadata # TODO : ajouter reference aux articles contenus dans ce chunk + lex id, langue et type ?
            )
        )

0it [00:00, ?it/s]

In [108]:
file_path = "chunks.txt"

with open(file_path, "w", encoding="utf-8") as f:
    for chunk in chunks:
        content = f"\n------------ DOC ID: {chunk.metadata["doc_id"]} - SOURCE: {chunk.metadata["source"]} - TOTAL PAGES: {chunk.metadata["total_pages"]} - START INDEX: {chunk.metadata["start_index"]} ------------\n"
        f.write(content + chunk.page_content + "\n")

In [101]:
if not os.getenv("BGE_API_KEY_EMBEDDINGS"):
    os.environ["BGE_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model (bge): ")

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("BGE_API_KEY_EMBEDDINGS")
)

sparse = FastEmbedSparse(model_name="Qdrant/bm42-all-minilm-l6-v2-attentions")
qdrant = QdrantVectorStore.from_documents(
    documents=chunks,
    url="http://localhost:6333",
    embedding=embeddings,
    sparse_embedding=sparse,
    collection_name="tika_bge_RCTS_window_hybrid",
    retrieval_mode=RetrievalMode.HYBRID
)

In [67]:
question = "Est-ce que le COSEC est rattaché à la VPF à l'EPFL ?"
dense_q = embeddings.embed_query(question)
sparse_q = sparse.embed_query(question)

In [68]:
print(dense_q)

[0.02620583400130272, 0.009765365161001682, -0.049907032400369644, 0.030271029099822044, -0.032790955156087875, -0.017861103639006615, 0.0202863197773695, -0.010672497563064098, -0.0018123290501534939, 0.011744238436222076, 0.006653918884694576, 0.002701715100556612, -0.03651628643274307, 0.04317424073815346, 0.020575933158397675, 0.024489115923643112, 0.05110461637377739, -0.010034637525677681, -0.00019793451065197587, 0.0011595221003517509, -0.02681715227663517, 0.0259409099817276, -0.013567754998803139, 0.037927836179733276, 0.006332699209451675, 0.018659165129065514, 0.03384116291999817, 0.033684808760881424, 0.02800910361111164, -0.03164828568696976, 0.006535313557833433, 0.018936268985271454, 0.011539367027580738, -0.020597193390130997, -0.011388164944946766, -0.003962765913456678, -0.01779581420123577, 0.028615806251764297, 0.00016090102144517004, 0.03756391257047653, 0.004225550219416618, 0.02732301689684391, 0.028125112876296043, -0.02981020137667656, 0.05670233815908432, -0.0

In [21]:
print(sparse_q)

indices=[390138254, 1162037697, 1024223489, 1006115573, 1775354856, 492661292, 111049216, 1177713239, 342821761, 1324212443, 1457734162] values=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [ ]:
# FIXME : old

from langchain_community.document_loaders import (
    PyMuPDFLoader,
    TextLoader,
    Docx2txtLoader,
    DirectoryLoader
)

loaders = [
    DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader, loader_kwargs={"mode": "single"}, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.txt", loader_cls=TextLoader, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.docx", loader_cls=Docx2txtLoader, show_progress=True),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())